# Impacts on Global Oil Trade

This notebook examines whether Singapore's oil trade shifted away from Middle East supply and toward alternative sources after geopolitical tensions around the Strait of Hormuz.

**Americas** is used as a proxy for US-linked supply, and **Africa** is used as a proxy for Cape of Good Hope / Africa-linked alternative routes.


## 1. Load Libraries

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = "{:,.2f}".format

## 2. Load Singapore Oil Import Data

In [7]:
# Detect project root from either the repository root or the Visualisation_Board folder.
cwd = Path.cwd()
if (cwd / "Data" / "raw").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "Data" / "raw").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path("/Users/veronica/Desktop/AI-Immigration-main")

RAW = PROJECT_ROOT / "Data" / "raw"

imports = pd.read_parquet(RAW / "singapore_imports_20260603_112515.parquet")
imports["date"] = pd.to_datetime(imports["date"])

imports.head()


,date,country,import_value
0,2019-01-01,Middle East,62.50
1,2019-02-01,Middle East,61.79
2,2019-03-01,Middle East,62.51
3,2019-04-01,Middle East,63.31
4,2019-05-01,Middle East,61.49


## 3. Descriptive Statistics

In [8]:
imports.groupby("country")["import_value"].describe()


,count,mean,std,min,25%,50%,75%,max
country,,,,,,,,
Africa,84.00,6.69,1.05,4.72,5.79,6.78,7.56,8.91
Americas,84.00,8.92,1.83,5.60,7.26,8.93,10.46,12.76
Middle East,84.00,58.99,1.91,54.20,57.59,58.95,60.31,63.31
Other Asia-Pacific,84.00,25.40,1.62,22.32,24.06,25.52,26.56,28.61


## 4. Main Visualization: Middle East vs Alternative Supply

In [9]:
alt_imports = (
    imports
    .assign(
        group=lambda df: np.where(
            df["country"].isin(["Americas", "Africa"]),
            "Americas + Africa alternative supply",
            df["country"],
        )
    )
    .groupby(["date", "group"], as_index=False)["import_value"]
    .sum()
)

focus = alt_imports[
    alt_imports["group"].isin([
        "Middle East",
        "Americas + Africa alternative supply",
    ])
]

fig = px.line(
    focus,
    x="date",
    y="import_value",
    color="group",
    markers=True,
    title="Singapore Oil Imports: Middle East vs Alternative Supply Proxy",
    labels={
        "date": "Month",
        "import_value": "Import share / index",
        "group": "Supply source",
    },
)

fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


## 5. Interpretation

The chart compares Singapore's Middle East oil imports with an alternative-supply proxy made from **Americas** and **Africa** imports.

If the alternative-supply line rises while Middle East imports decline, this suggests that Singapore may be diversifying away from Gulf/Hormuz-linked supply toward other exporters or routes, including US-linked supply and Africa/Cape-linked routes.

This does not directly prove individual ship rerouting through Houston or the Cape of Good Hope because the dataset does not include direct AIS route tracks. However, it provides useful trade-flow evidence for the broader project question: **how geopolitical conflict reshapes global oil trade routes**.
